In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

In [4]:
from google.colab import drive
drive.mount('/content/drive/MyDrive/Boatdataset.zip')

ValueError: Mountpoint must be in a directory that exists

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Lung dataset (Class).zip'
extract_path = '/content/data/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")

Extraction complete!


In [9]:
data_dir = '/content/data/lung dataset'

In [19]:
batch_size = 32
img_height = 180
img_width = 180
epochs = 2

In [11]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 13808 files belonging to 2 classes.
Using 11047 files for training.


In [12]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 13808 files belonging to 2 classes.
Using 2761 files for validation.


In [13]:
train_ds.class_names

['covid', 'normal']

In [14]:
val_ds.class_names

['covid', 'normal']

In [15]:
model = Sequential(
    [
        layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(img_height, img_width, 3)),

        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),


        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 180, 180, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 180, 180, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 90, 90, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 90, 90, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 90, 90, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 45, 45, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 45, 45, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 45, 45, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 22, 22, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 61952)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     7,929,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,060,289 (30.75 MB)

 Trainable params: 8,060,289 (30.75 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [20]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/nn.py:1286: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


346/346 ━━━━━━━━━━━━━━━━━━━━ 2850s 8s/step - accuracy: 0.7992 - loss: 2.0488 - val_accuracy: 0.8783 - val_loss: 0.2836
Epoch 2/2
346/346 ━━━━━━━━━━━━━━━━━━━━ 2970s 8s/step - accuracy: 0.8964 - loss: 0.2465 - val_accuracy: 0.8924 - val_loss: 0.2467


In [21]:
#predicting

from tensorflow.keras.preprocessing import image
import numpy as np

# Load the Image and preprocess it
def preprocess_image(img_path, img_height, img_width):
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    print(img_array.shape)
    return img_array

In [24]:
# Path to your image
img_path = '/content/COVID-1266.png'
img_path2 = '/content/Normal-104.png'

# Preprocess the image
img_array = preprocess_image(img_path, img_height, img_width)
img_array2 = preprocess_image(img_path2, img_height, img_width)

# Predict

prediction = model.predict(img_array)
prediction2 = model.predict(img_array2)

print(prediction)
print(prediction2)

(1, 180, 180, 3)
(1, 180, 180, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
[[0.46585098]]
[[0.98963994]]


In [26]:
# Convert prediction to class level
def diagnose(prediction):
    if prediction[0][0] > 0.5:
      print("predicted class: 1 (Positive)")
    else:
      print("predicted class: 0 (Negative)")
diagnose(prediction)
diagnose(prediction2)

predicted class: 0 (Negative)
predicted class: 1 (Positive)


Pre-Trained Models


1. Inception
2. MobileNet
3. Res-Net
4. VGG-16

In [27]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2


In [30]:
base_model = MobileNetV2(weights='imagenet', input_shape= (img_height,  img_width, 3), include_top=False)

/tmp/ipykernel_5334/1779559720.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', input_shape= (img_height,  img_width, 3), include_top=False)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [32]:
model2 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Not traning the model from scratch
base_model.trainable = False


model2.compile(optimizer=tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=['accuracy'])

epochs  = 2

history = model2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

Epoch 1/2
346/346 ━━━━━━━━━━━━━━━━━━━━ 496s 1s/step - accuracy: 0.8216 - loss: 0.4015 - val_accuracy: 0.8819 - val_loss: 0.3042
Epoch 2/2
346/346 ━━━━━━━━━━━━━━━━━━━━ 490s 1s/step - accuracy: 0.8683 - loss: 0.3036 - val_accuracy: 0.8674 - val_loss: 0.2960


# Predicting If a Lungs have covid or not


In [33]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the Image and preprocess it
def preprocess_image(img_path, img_height, img_width):
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    print(img_array.shape)
    return img_array

In [34]:
# Path to your image
img_path = '/content/COVID-1266.png'
img_path2 = '/content/Normal-104.png'

# Preprocess the image
img_array = preprocess_image(img_path, img_height, img_width)
img_array2 = preprocess_image(img_path2, img_height, img_width)

# Predict

prediction = model2.predict(img_array)
prediction2 = model2.predict(img_array2)

print(prediction)
print(prediction2)

(1, 180, 180, 3)
(1, 180, 180, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
[[0.61862]]
[[0.9995077]]


In [35]:
# Convert prediction to class level
def diagnose(prediction):
    if prediction[0][0] > 0.5:
      print("predicted class: 1 (Positive)")
    else:
      print("predicted class: 0 (Negative)")
diagnose(prediction)
diagnose(prediction2)

predicted class: 1 (Positive)
predicted class: 1 (Positive)
